In [ ]:
%%capture
# Updated: January 2026 - Latest package versions
# Note: If you've already run install.sh, llama-index is already installed
!pip install llama-index==0.14.13 html2text

In [1]:
# Standard library imports
import os
from getpass import getpass
import nest_asyncio

# Third-party imports
from dotenv import load_dotenv

# Apply nest_asyncio to allow nested event loops (needed for Jupyter notebooks)
# This is required when using async operations in Jupyter
nest_asyncio.apply()

# Load environment variables from .env file
# This will read API keys and other variables from the .env file in the project root
load_dotenv()

True

# 📂 **Loading Data**

Preparing your data for an LLM involves an ingestion pipeline similar to ML data cleaning or traditional ETL processes.

### **Ingestion Pipeline Stages**
  - 📥 Load the data
  - 🔧 Transform the data
  - 🗃️ Index and store the data


Let's start by downloading some example files

In [28]:
# Import required libraries
import requests
from pathlib import Path
from time import sleep

# Base URL for Project Gutenberg texts
# Project Gutenberg provides free ebooks - we'll download text versions
base_url = "https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"

# Directory to save the downloaded files
# Using ../data to match the data directory structure from Module 1
directory = Path("../dataM2")

# Create the directory if it doesn't exist
# parents=True creates parent directories if needed, exist_ok=True doesn't error if it exists
directory.mkdir(parents=True, exist_ok=True)

# Generate a list of book IDs to download
# Project Gutenberg book IDs - we'll download books 1-10 as examples
book_ids = range(1, 11)  # This will create a range from 1 to 10

# Generate URLs for each book ID using list comprehension
# This creates a list of URLs by formatting the base_url with each book_id
urls = [base_url.format(book_id=book_id) for book_id in book_ids]

# Download each file and save it in the specified directory
# Loop through each URL and download the file
for url in urls:
    try:
        # Make HTTP GET request to download the file
        # Add timeout to prevent hanging indefinitely
        response = requests.get(url, timeout=30)
        
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            # Extract the filename from the URL using the book ID
            # URL format: .../epub/{book_id}/pg{book_id}.txt
            book_id = url.split('/')[-2]  # Extracts the book ID from the URL
            filename = f"pg{book_id}.txt"
            file_path = directory / filename
            
            # Save the file to the specified directory
            # write_text() writes the response text content to the file
            file_path.write_text(response.text, encoding='utf-8')
            print(f"Downloaded {filename} to {file_path}")
        else:
            print(f"Failed to download {url}. HTTP status code: {response.status_code}")
    except requests.exceptions.RequestException as e:
        # Handle network errors gracefully
        print(f"Error downloading {url}: {e}")

Downloaded pg1.txt to ..\dataM2\pg1.txt
Downloaded pg2.txt to ..\dataM2\pg2.txt
Downloaded pg3.txt to ..\dataM2\pg3.txt
Downloaded pg4.txt to ..\dataM2\pg4.txt
Downloaded pg5.txt to ..\dataM2\pg5.txt
Downloaded pg6.txt to ..\dataM2\pg6.txt
Downloaded pg7.txt to ..\dataM2\pg7.txt
Downloaded pg8.txt to ..\dataM2\pg8.txt
Downloaded pg9.txt to ..\dataM2\pg9.txt
Downloaded pg10.txt to ..\dataM2\pg10.txt


# 📥 Load the data

To use data with an LLM, first load it using data connectors, known as `Readers` in LlamaIndex, which format data into `Document` objects containing data and metadata.

📚 **SimpleDirectoryReader**:
  - The most straightforward loader is `SimpleDirectoryReader``.
  - Built into LlamaIndex, it reads various formats (Markdown, PDFs, Word documents, PowerPoint decks, images, audio, video) from every file in a directory, creating documents.

In [29]:
# Import SimpleDirectoryReader from LlamaIndex core
# SimpleDirectoryReader is a data connector that reads files from a directory
from llama_index.core import SimpleDirectoryReader

# Load all documents from the data directory
# SimpleDirectoryReader automatically detects and reads various file formats:
# - Text files (.txt)
# - PDFs (.pdf)s
# - Markdown (.md)
# - Word documents (.docx)
# - PowerPoint (.pptx)
# - Images, audio, video files
# load_data() returns a list of Document objects
# Note: Path should match where we saved the downloaded files (../dataM2)
documents = SimpleDirectoryReader("../dataM2").load_data()

In [30]:
# Check how many documents were loaded
# This tells us how many files were successfully read from the directory
len(documents)

10

In [32]:
# Check the type of the first document
# Should be llama_index.core.schema.Document
# Document objects contain text content and metadata
type(documents[0])

llama_index.core.schema.Document

In [33]:
# Inspect the internal structure of a document
# __dict__ shows all attributes of the Document object, including:
# - text: The actual content
# - metadata: File information (filename, file path, etc.)
# - id_: Unique identifier
# - relationships: Connections to other nodes/documents
documents[3].__dict__

{'id_': '09e65a55-fafd-470e-8985-3b8ba723cd0a',
 'embedding': None,
 'metadata': {'file_path': 'c:\\Users\\owais\\Documents\\AI\\hands-on-ai-rag-using-llamaindex-3830207\\02_Fundamental_Concepts_in_LlamaIndex\\..\\dataM2\\pg3.txt',
  'file_name': 'pg3.txt',
  'file_type': 'text/plain',
  'file_size': 28163,
  'creation_date': '2026-01-27',
  'last_modified_date': '2026-01-27'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='\ufeffThe Project Gutenberg eBook of John F. Kennedy\'s Inaugural Address\r\r\n    \r\r\nThis ebook is for the use of anyone anywhere in the United States and\r\r\nmost other

##### Manually Create Document Objects

In [34]:
# Import Document class from LlamaIndex core
# Document is the base class for all text data in LlamaIndex
from llama_index.core import Document

# Create a Document object manually
# This is useful when you want to create documents programmatically
# rather than loading from files
manual_document = Document(text="This is an example of a manual document")

In [35]:
# Inspect the manually created document's structure
# Notice it has the same structure as documents loaded from files
# but without file-related metadata
manual_document.__dict__

{'id_': '759b18ca-3503-472a-a5fa-03440da25093',
 'embedding': None,
 'metadata': {},
 'excluded_embed_metadata_keys': [],
 'excluded_llm_metadata_keys': [],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='This is an example of a manual document', path=None, url=None, mimetype=None),
 'image_resource': None,
 'audio_resource': None,
 'video_resource': None,
 'text_template': '{metadata_str}\n\n{content}'}

##### Adding metadata

You can add metadata in the document constructor:

In [36]:
# Create a Document with custom metadata
# Metadata is a dictionary that can store any key-value pairs
# Common metadata includes: filename, category, author, date, source, etc.
# Metadata is useful for filtering, organizing, and retrieving documents
manual_document_with_metadata = Document(
    text="This is an example of a manual document",
    metadata={"filename": "made-up-file-name", "category": "imaginary-category"}
)

In [37]:
# Inspect the document with metadata
# Notice the metadata dictionary is now populated with our custom values
manual_document_with_metadata.__dict__

{'id_': '70f26a8d-e54d-4679-a2fa-07fb548b937c',
 'embedding': None,
 'metadata': {'filename': 'made-up-file-name',
  'category': 'imaginary-category'},
 'excluded_embed_metadata_keys': [],
 'excluded_llm_metadata_keys': [],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='This is an example of a manual document', path=None, url=None, mimetype=None),
 'image_resource': None,
 'audio_resource': None,
 'video_resource': None,
 'text_template': '{metadata_str}\n\n{content}'}

Or after the document is created

In [38]:
# Add or update metadata after document creation
# You can modify metadata at any time by assigning a new dictionary
# This is useful when you need to add metadata based on processing results
manual_document.metadata = {"filename": "made-up-file-name", "category": "imaginary-category"}

In [39]:
manual_document.__dict__

{'id_': '759b18ca-3503-472a-a5fa-03440da25093',
 'embedding': None,
 'metadata': {'filename': 'made-up-file-name',
  'category': 'imaginary-category'},
 'excluded_embed_metadata_keys': [],
 'excluded_llm_metadata_keys': [],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='This is an example of a manual document', path=None, url=None, mimetype=None),
 'image_resource': None,
 'audio_resource': None,
 'video_resource': None,
 'text_template': '{metadata_str}\n\n{content}'}

# 🔧 Transform the data

After loading, we must process and transform data for retrieval. We need to transform the list of `Document` objects into `Node` objects 

- ✂️ Include chunking, extracting metadata, and embedding each chunk in transformations.

- 🌟 Nodes are a first-class citizen in LlamaIndex, allowing direct definition or parsing from Documents.

- 🔄 Transformation inputs and outputs are `Node` objects (Note: `Document` is subclass of `Node`).

- 🛠️ Nodes are "chunks" of Documents, including text, images, etc., plus metadata and relationships.

- 📊 `NodeParser` classes convert Documents into Nodes with all necessary attributes. There are [a number of](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/modules.html) `NodeParser`'s you can choose from!

- 📑 High-level API: Use `.from_documents()` for automatic parsing and chunking of Document objects.

- 🔍 Underlying process splits Document into Node objects, maintaining text and metadata with a link to their parent Document.


In [40]:
# Import SentenceSplitter from LlamaIndex
# SentenceSplitter is a NodeParser that splits documents into smaller chunks (nodes)
from llama_index.core.node_parser import SentenceSplitter

# Create a SentenceSplitter parser with custom settings
# chunk_size: Maximum size of each chunk in tokens (not characters)
# chunk_overlap: Number of tokens to overlap between chunks (helps maintain context)
# paragraph_separator: Prefer splitting at paragraph boundaries when possible
parser = SentenceSplitter(
    chunk_size=128,  # in tokens - smaller chunks = more granular retrieval
    chunk_overlap=16,  # in tokens - overlap prevents losing context at boundaries
    paragraph_separator="\n\n"  # Split at double newlines (paragraph breaks)
)

# Convert Documents into Nodes
# get_nodes_from_documents() processes all documents and splits them into nodes
# show_progress=True displays a progress bar during processing
# Nodes are smaller chunks of the original documents, ready for indexing
nodes = parser.get_nodes_from_documents(documents, show_progress=True)

Parsing nodes:   0%|          | 0/10 [00:00<?, ?it/s]

In [41]:
# Check the type of a node
# Should be llama_index.core.schema.TextNode (or similar Node subclass)
# Nodes are chunks of documents with additional metadata and relationships
type(nodes[42])

llama_index.core.schema.TextNode

In [42]:
# Inspect the internal structure of a node
# Nodes contain:
# - text: The chunked text content
# - metadata: Inherited from parent document + chunk-specific metadata
# - id_: Unique identifier for this node
# - relationships: Links to parent document, next/previous nodes, etc.
# - embedding: Vector representation (added during indexing)
nodes[42].__dict__

{'id_': '4b8585c2-3840-4b0c-b374-d04664e9c57a',
 'embedding': None,
 'metadata': {'file_path': 'c:\\Users\\owais\\Documents\\AI\\hands-on-ai-rag-using-llamaindex-3830207\\02_Fundamental_Concepts_in_LlamaIndex\\..\\dataM2\\pg1.txt',
  'file_name': 'pg1.txt',
  'file_type': 'text/plain',
  'file_size': 31482,
  'creation_date': '2026-01-27',
  'last_modified_date': '2026-01-27'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='cfef1682-216d-4b10-9798-3381dfbe2160', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\owais\\Documents\\AI\\hands-on-ai-rag-using-llamaindex-3830207\\02_Fundamental_Concepts_in_LlamaIndex\\..\\dataM2\\pg1.txt', 'file_name': 'pg1.

You can also choose to construct Node objects manually.


In [43]:
# Import classes for manually creating nodes with relationships
# TextNode: A node containing text content
# NodeRelationship: Enumeration of relationship types (PARENT, CHILD, NEXT, PREVIOUS, etc.)
# RelatedNodeInfo: Information about a related node (ID and optional metadata)
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo

# Create TextNode objects manually
# id_ parameter assigns a unique identifier to each node
# This is useful when you want explicit control over node creation
node1 = TextNode(text="Dad is married to Mom", id_="001")

node2 = TextNode(text="Dad is Son's dad", id_="002")

## NodeRelationships

You can set relationships between nodes.

- 🌐 NodeRelationships assign connections between chunks of text. It's useful for:
  - Documents organized in a hierarchical manner (e.g., book, chapter, section, subsection)
  - Maintaining sequential order
  - Other complex relationships (ie, in legal documents for links a clause or other cases) 

- 🔍 NodeRelationships help retrieve not just the relevant section, but also related sections that might provide additional context or information.

In [44]:
# Establish relationships between nodes
# Relationships help maintain structure and context between chunks

# Set NEXT relationship: node1 comes before node2
# This creates a sequential link, useful for maintaining order
node1.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
    node_id=node2.node_id
)

# Set PREVIOUS relationship: node2 comes after node1
# This creates a bidirectional link for navigation
node2.relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(
    node_id=node1.node_id
)

# Create a list of nodes (for use in indexing)
nodes = [node1, node2]

# Set PARENT relationship: node1 is the parent of node2
# PARENT relationships are useful for hierarchical structures (e.g., chapter -> section)
# You can also add metadata to relationships for additional context
node2.relationships[NodeRelationship.PARENT] = RelatedNodeInfo(
    node_id=node1.node_id, 
    metadata={"Romie": "Mom", "Harpreet": "Dad", "Jind": "Daughter", "Jugaad": "Son"}
)

A bit of clean up, let's just go ahead and delete the text files we downloaded since we won't need them going forward.


In [ ]:
# Clean up: Delete the downloaded text files
# Note: This command works on Unix/Linux/Mac. On Windows, use: !rmdir /s /q data
# Alternatively, use Python's pathlib for cross-platform deletion:
# import shutil
# shutil.rmtree("../data", ignore_errors=True)

# For Unix/Linux/Mac:
!rm -rf ../dataM2

# For Windows (uncomment if needed):
# !rmdir /s /q ..\\data